# GeoBinV4 Buffer Overflow Hypothesis Analysis

Comparing two simulation runs at 200 satellites:
- **Old run (100 GB buffer):** `maxdownload_20260218_204617`
- **New run (10 TB buffer):** `maxdownload_20260219_072342`

Goal: Determine if the 100 GB buffer was causing overflow/data loss.

In [1]:
import os
import glob
import zipfile
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

BASE = '/Users/chrischeshire/github/cote-constellation/examples/bent-pipe-constellation/'
OLD_RUN = os.path.join(BASE, 'results/maxdownload_20260218_204617')
NEW_RUN = os.path.join(BASE, 'results/maxdownload_20260219_072342')

print("Old run contents:", os.listdir(OLD_RUN))
print("New run contents:", [x for x in os.listdir(NEW_RUN) if not x.startswith('.')])

Old run contents: ['constellation_analysis_20260218_214508_28000_200', 'constellation_analysis_20260218_204641_00279_200']
New run contents: ['constellation_analysis_20260219_072406_02799_200', 'constellation_analysis_20260219_082538_28000_200']


## Locate All Zip File Paths

In [2]:
# Define all zip paths using exact directory names found above
zip_paths = {
    'Old 28MB close': os.path.join(OLD_RUN, 'constellation_analysis_20260218_214508_28000_200/close-spaced/simulation_logs.zip'),
    'Old 28MB orbit': os.path.join(OLD_RUN, 'constellation_analysis_20260218_214508_28000_200/orbit-spaced/simulation_logs.zip'),
    'New 28MB close': os.path.join(NEW_RUN, 'constellation_analysis_20260219_082538_28000_200/close-spaced/simulation_logs.zip'),
    'New 28MB orbit': os.path.join(NEW_RUN, 'constellation_analysis_20260219_082538_28000_200/orbit-spaced/simulation_logs.zip'),
    'New 2.8MB close': os.path.join(NEW_RUN, 'constellation_analysis_20260219_072406_02799_200/close-spaced/simulation_logs.zip'),
    'New 2.8MB orbit': os.path.join(NEW_RUN, 'constellation_analysis_20260219_072406_02799_200/orbit-spaced/simulation_logs.zip'),
}

for label, path in zip_paths.items():
    exists = os.path.isfile(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else 0
    print(f"{'✓' if exists else '✗'} {label}: {size_mb:.1f} MB  ({path})")

✓ Old 28MB close: 46.6 MB  (/Users/chrischeshire/github/cote-constellation/examples/bent-pipe-constellation/results/maxdownload_20260218_204617/constellation_analysis_20260218_214508_28000_200/close-spaced/simulation_logs.zip)
✓ Old 28MB orbit: 59.3 MB  (/Users/chrischeshire/github/cote-constellation/examples/bent-pipe-constellation/results/maxdownload_20260218_204617/constellation_analysis_20260218_214508_28000_200/orbit-spaced/simulation_logs.zip)
✓ New 28MB close: 51.3 MB  (/Users/chrischeshire/github/cote-constellation/examples/bent-pipe-constellation/results/maxdownload_20260219_072342/constellation_analysis_20260219_082538_28000_200/close-spaced/simulation_logs.zip)
✓ New 28MB orbit: 65.1 MB  (/Users/chrischeshire/github/cote-constellation/examples/bent-pipe-constellation/results/maxdownload_20260219_072342/constellation_analysis_20260219_082538_28000_200/orbit-spaced/simulation_logs.zip)
✓ New 2.8MB close: 52.1 MB  (/Users/chrischeshire/github/cote-constellation/examples/bent-pi

## Helper Functions

In [3]:
def list_zip_contents(zip_path):
    """List all files inside a zip archive."""
    with zipfile.ZipFile(zip_path, 'r') as zf:
        return zf.namelist()

def read_csv_from_zip(zip_path, csv_inner_path):
    """Read a CSV file from inside a zip archive into a DataFrame."""
    with zipfile.ZipFile(zip_path, 'r') as zf:
        names = zf.namelist()
        # Try exact match first, then partial match
        if csv_inner_path in names:
            with zf.open(csv_inner_path) as f:
                return pd.read_csv(io.BytesIO(f.read()))
        # Try finding it with different prefix
        matches = [n for n in names if n.endswith(csv_inner_path) or csv_inner_path in n]
        if matches:
            with zf.open(matches[0]) as f:
                return pd.read_csv(io.BytesIO(f.read()))
        raise FileNotFoundError(f"'{csv_inner_path}' not found in zip. Available: {[n for n in names if n.endswith('.csv')]}")

def check_buffer_overflow(zip_path):
    """Search for buffer overflow related files in the zip."""
    names = list_zip_contents(zip_path)
    overflow_files = [n for n in names if any(kw in n.lower() for kw in ['overflow', 'buffer', 'drop', 'lost'])]
    return overflow_files

def compute_5deg_bins(df, lat_col='lat', lon_col='lon'):
    """Bin lat/lon into 5-degree cells and count unique bins."""
    lat_bins = np.floor(df[lat_col].values / 5) * 5
    lon_bins = np.floor(df[lon_col].values / 5) * 5
    unique_bins = set(zip(lat_bins, lon_bins))
    return len(unique_bins), unique_bins

def analyze_zip(label, zip_path):
    """Full analysis of one simulation zip file. Returns a dict of metrics."""
    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"  {zip_path}")
    print(f"{'='*70}")
    
    results = {'label': label}
    
    # List contents
    contents = list_zip_contents(zip_path)
    csv_files = sorted([f for f in contents if f.endswith('.csv')])
    print(f"\nCSV files in zip ({len(csv_files)}):")
    for f in csv_files:
        print(f"  {f}")
    
    # Check for buffer overflow files
    overflow_files = check_buffer_overflow(zip_path)
    results['overflow_files'] = overflow_files
    if overflow_files:
        print(f"\n⚠️  BUFFER OVERFLOW FILES FOUND: {overflow_files}")
        for of in overflow_files:
            try:
                df_ov = read_csv_from_zip(zip_path, of)
                print(f"  {of}: {len(df_ov)} rows")
                print(df_ov.head())
            except:
                pass
    else:
        print(f"\n✓ No buffer overflow files found")
    
    # Image completions
    try:
        df_comp = read_csv_from_zip(zip_path, 'geobinv4/image_completions.csv')
        results['total_completions'] = len(df_comp)
        results['unique_bin_ids'] = df_comp['bin_id'].nunique() if 'bin_id' in df_comp.columns else None
        
        if 'lat' in df_comp.columns and 'lon' in df_comp.columns:
            n_bins, bins_set = compute_5deg_bins(df_comp)
            results['download_5deg_bins'] = n_bins
            results['download_bins_set'] = bins_set
        else:
            results['download_5deg_bins'] = None
            results['download_bins_set'] = set()
        
        print(f"\n📥 Image Completions: {len(df_comp)} total downloads")
        print(f"   Columns: {list(df_comp.columns)}")
        if 'bin_id' in df_comp.columns:
            print(f"   Unique bin_ids: {df_comp['bin_id'].nunique()}")
        if 'bin_count' in df_comp.columns:
            print(f"   bin_count range: {df_comp['bin_count'].min()} - {df_comp['bin_count'].max()}, mean={df_comp['bin_count'].mean():.1f}")
        if results['download_5deg_bins'] is not None:
            print(f"   Unique 5-deg bins (downloads): {results['download_5deg_bins']}")
        print(f"   First few rows:")
        print(df_comp.head(3).to_string())
    except Exception as e:
        print(f"\n❌ Image completions error: {e}")
        results['total_completions'] = 0
        results['download_5deg_bins'] = 0
        results['download_bins_set'] = set()
    
    # Visibility log
    try:
        df_vis = read_csv_from_zip(zip_path, 'geobinv4/visibility_log.csv')
        print(f"\n👁️  Visibility Log: {len(df_vis)} total rows")
        print(f"   Columns: {list(df_vis.columns)}")
        
        # Image taken
        if 'image_taken' in df_vis.columns:
            images_captured = (df_vis['image_taken'] == 1).sum()
            results['images_captured'] = int(images_captured)
            print(f"   Images captured (image_taken==1): {images_captured}")
            
            # 5-deg bins for captures
            df_captured = df_vis[df_vis['image_taken'] == 1]
            if 'lat' in df_captured.columns and 'lon' in df_captured.columns and len(df_captured) > 0:
                n_obs_bins, obs_bins_set = compute_5deg_bins(df_captured)
                results['observed_5deg_bins'] = n_obs_bins
                results['observed_bins_set'] = obs_bins_set
                print(f"   Unique 5-deg bins (observed/captured): {n_obs_bins}")
            else:
                results['observed_5deg_bins'] = None
                results['observed_bins_set'] = set()
        else:
            results['images_captured'] = None
            results['observed_5deg_bins'] = None
            results['observed_bins_set'] = set()
        
        # Downloaded
        if 'downloaded_mb' in df_vis.columns:
            downloads_positive = (df_vis['downloaded_mb'] > 0).sum()
            total_downloaded_mb = df_vis['downloaded_mb'].sum()
            results['visibility_downloads'] = int(downloads_positive)
            results['total_downloaded_mb'] = float(total_downloaded_mb)
            print(f"   Rows with downloaded_mb > 0: {downloads_positive}")
            print(f"   Total downloaded: {total_downloaded_mb:.1f} MB")
        else:
            results['visibility_downloads'] = None
            results['total_downloaded_mb'] = None
        
        print(f"   First few rows:")
        print(df_vis.head(3).to_string())
    except Exception as e:
        print(f"\n❌ Visibility log error: {e}")
        results['images_captured'] = None
        results['observed_5deg_bins'] = None
        results['observed_bins_set'] = set()
        results['visibility_downloads'] = None
        results['total_downloaded_mb'] = None
    
    # Coverage ratio
    dl_bins = results.get('download_5deg_bins')
    obs_bins = results.get('observed_5deg_bins')
    if dl_bins is not None and obs_bins is not None and obs_bins > 0:
        results['coverage_ratio'] = dl_bins / obs_bins
        print(f"\n📊 Coverage: {dl_bins}/{obs_bins} = {results['coverage_ratio']:.4f} ({results['coverage_ratio']*100:.2f}%)")
    else:
        results['coverage_ratio'] = None
    
    return results

print("Helper functions defined ✓")

Helper functions defined ✓


## Run Analysis on All 6 Configurations

In [ ]:
# Run analysis on all 6 zip files
all_results = {}
for label, path in zip_paths.items():
    all_results[label] = analyze_zip(label, path)


  Old 28MB close
  /Users/chrischeshire/github/cote-constellation/examples/bent-pipe-constellation/results/maxdownload_20260218_204617/constellation_analysis_20260218_214508_28000_200/close-spaced/simulation_logs.zip

CSV files in zip (405):
  geobinv4/evnt-trigger-time.csv
  geobinv4/image_completions.csv
  geobinv4/meas-MB-buffered-sat-0060518000.csv
  geobinv4/meas-MB-buffered-sat-0060518001.csv
  geobinv4/meas-MB-buffered-sat-0060518002.csv
  geobinv4/meas-MB-buffered-sat-0060518003.csv
  geobinv4/meas-MB-buffered-sat-0060518004.csv
  geobinv4/meas-MB-buffered-sat-0060518005.csv
  geobinv4/meas-MB-buffered-sat-0060518006.csv
  geobinv4/meas-MB-buffered-sat-0060518007.csv
  geobinv4/meas-MB-buffered-sat-0060518008.csv
  geobinv4/meas-MB-buffered-sat-0060518009.csv
  geobinv4/meas-MB-buffered-sat-0060518010.csv
  geobinv4/meas-MB-buffered-sat-0060518011.csv
  geobinv4/meas-MB-buffered-sat-0060518012.csv
  geobinv4/meas-MB-buffered-sat-0060518013.csv
  geobinv4/meas-MB-buffered-sat-0

/var/folders/0v/5c9vlpns3w5c58y_r464t87r0000gp/T/ipykernel_19423/2482020434.py:13: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.BytesIO(f.read()))



👁️  Visibility Log: 1663487 total rows
   Columns: ['time', 'sat_id', 'in_view', 'connected', 'buffer_mb', 'downloaded_mb', 'image_taken', 'lat_deg', 'lon_deg', 'freshness_timestamp', 'distance_km', 'elevation_deg', 'decision_interval', 'bitrate_mbps', 'image_completed', 'completed_image_lat', 'completed_image_lon', 'completed_image_timestamp', 'downloaded_image_lat', 'downloaded_image_lon']
   Images captured (image_taken==1): 1440000
   Rows with downloaded_mb > 0: 1661
   Total downloaded: 23561.7 MB
   First few rows:
   time    sat_id  in_view  connected  buffer_mb  downloaded_mb  image_taken    lat_deg     lon_deg            freshness_timestamp  distance_km  elevation_deg  decision_interval  bitrate_mbps  image_completed  completed_image_lat  completed_image_lon completed_image_timestamp  downloaded_image_lat  downloaded_image_lon
0   2.0  60518000        0          0        0.0            0.0            1  70.980644 -235.768995  2025-08-27T13:03:22.698930000          0.0       

## Unified Comparison Table

In [ ]:
# Build comparison table
rows = []
config_meta = {
    'Old 28MB close': ('100 GB', '28 MB', 'close-spaced'),
    'Old 28MB orbit': ('100 GB', '28 MB', 'orbit-spaced'),
    'New 28MB close': ('10 TB', '28 MB', 'close-spaced'),
    'New 28MB orbit': ('10 TB', '28 MB', 'orbit-spaced'),
    'New 2.8MB close': ('10 TB', '2.8 MB', 'close-spaced'),
    'New 2.8MB orbit': ('10 TB', '2.8 MB', 'orbit-spaced'),
}

for label in zip_paths.keys():
    r = all_results[label]
    buf, img_sz, spacing = config_meta[label]
    rows.append({
        'Config': label,
        'Buffer': buf,
        'Image Size': img_sz,
        'Spacing': spacing,
        'Images Captured': r.get('images_captured'),
        'Total Completions': r.get('total_completions'),
        'Vis Downloads': r.get('visibility_downloads'),
        'Total Downloaded (MB)': f"{r.get('total_downloaded_mb', 0):.1f}" if r.get('total_downloaded_mb') is not None else 'N/A',
        '5° Bins (Downloads)': r.get('download_5deg_bins'),
        '5° Bins (Observed)': r.get('observed_5deg_bins'),
        'Coverage Ratio': f"{r.get('coverage_ratio', 0):.4f}" if r.get('coverage_ratio') is not None else 'N/A',
        'Overflow Files': 'YES' if r.get('overflow_files') else 'No',
    })

df_table = pd.DataFrame(rows)
display(df_table)

# Also display as formatted HTML for better readability
print("\n\n" + "="*80)
print("  BUFFER OVERFLOW HYPOTHESIS COMPARISON")
print("="*80)
print(f"\nKey question: Does the 10 TB buffer run show MORE downloads than the 100 GB buffer run?")
print(f"If yes → buffer overflow was limiting performance in the old run.\n")

for label in ['Old 28MB close', 'New 28MB close', 'Old 28MB orbit', 'New 28MB orbit']:
    r = all_results[label]
    print(f"  {label:20s}: {r.get('total_completions', 'N/A'):>8} completions, "
          f"{r.get('download_5deg_bins', 'N/A'):>5} 5° bins, "
          f"coverage={r.get('coverage_ratio', 0):.4f}" if r.get('coverage_ratio') else f"  {label:20s}: N/A")

print(f"\n{'─'*80}")
print("28 MB Close-Spaced Delta:")
old_c = all_results['Old 28MB close'].get('total_completions', 0) or 0
new_c = all_results['New 28MB close'].get('total_completions', 0) or 0
if old_c > 0:
    pct = (new_c - old_c) / old_c * 100
    print(f"  Completions: {old_c} → {new_c} ({pct:+.1f}%)")
old_b = all_results['Old 28MB close'].get('download_5deg_bins', 0) or 0
new_b = all_results['New 28MB close'].get('download_5deg_bins', 0) or 0
if old_b > 0:
    pct_b = (new_b - old_b) / old_b * 100
    print(f"  5° bins:     {old_b} → {new_b} ({pct_b:+.1f}%)")

print(f"\n28 MB Orbit-Spaced Delta:")
old_o = all_results['Old 28MB orbit'].get('total_completions', 0) or 0
new_o = all_results['New 28MB orbit'].get('total_completions', 0) or 0
if old_o > 0:
    pct_o = (new_o - old_o) / old_o * 100
    print(f"  Completions: {old_o} → {new_o} ({pct_o:+.1f}%)")
old_bo = all_results['Old 28MB orbit'].get('download_5deg_bins', 0) or 0
new_bo = all_results['New 28MB orbit'].get('download_5deg_bins', 0) or 0
if old_bo > 0:
    pct_bo = (new_bo - old_bo) / old_bo * 100
    print(f"  5° bins:     {old_bo} → {new_bo} ({pct_bo:+.1f}%)")

## Visualization

In [ ]:
# Visualization: compare key metrics across all 6 configurations
labels = list(zip_paths.keys())
completions = [all_results[l].get('total_completions', 0) or 0 for l in labels]
dl_bins = [all_results[l].get('download_5deg_bins', 0) or 0 for l in labels]
obs_bins = [all_results[l].get('observed_5deg_bins', 0) or 0 for l in labels]
coverage = [all_results[l].get('coverage_ratio', 0) or 0 for l in labels]
captured = [all_results[l].get('images_captured', 0) or 0 for l in labels]
dl_mb = [float(all_results[l].get('total_downloaded_mb', 0) or 0) for l in labels]

# Color coding: old=red tones, new 28MB=blue tones, new 2.8MB=green tones
colors = ['#d62728', '#d62728', '#1f77b4', '#1f77b4', '#2ca02c', '#2ca02c']
hatches = ['', '//', '', '//', '', '//']  # solid=close, hatched=orbit

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('GeoBinV4 Buffer Overflow Hypothesis: 100 GB vs 10 TB Buffer\n(200 satellites, maxdownload scenario)', 
             fontsize=14, fontweight='bold')

# Panel 1: Total Completions
ax = axes[0, 0]
bars = ax.bar(range(len(labels)), completions, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Total Image Completions')
ax.set_title('Total Downloads (Image Completions)')
for i, v in enumerate(completions):
    ax.text(i, v + max(completions)*0.01, str(v), ha='center', va='bottom', fontsize=8)

# Panel 2: Coverage Ratio
ax = axes[0, 1]
bars = ax.bar(range(len(labels)), [c*100 for c in coverage], color=colors, edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Coverage (%)')
ax.set_title('Coverage Ratio (Download Bins / Observed Bins)')
for i, v in enumerate(coverage):
    ax.text(i, v*100 + max(coverage)*100*0.01, f'{v*100:.1f}%', ha='center', va='bottom', fontsize=8)

# Panel 3: Total Downloaded MB
ax = axes[1, 0]
bars = ax.bar(range(len(labels)), dl_mb, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Total Downloaded (MB)')
ax.set_title('Total Data Downloaded')
for i, v in enumerate(dl_mb):
    ax.text(i, v + max(dl_mb)*0.01, f'{v:.0f}', ha='center', va='bottom', fontsize=8)

# Panel 4: 5-deg bins observed vs downloaded
ax = axes[1, 1]
x = np.arange(len(labels))
width = 0.35
bars1 = ax.bar(x - width/2, obs_bins, width, label='Observed (captured)', color=[c+'80' for c in colors], edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, dl_bins, width, label='Downloaded', color=colors, edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Unique 5° Bins')
ax.set_title('5-Degree Bin Coverage (Observed vs Downloaded)')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(BASE, 'buffer_overflow_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to buffer_overflow_comparison.png")